<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/04_evaluation_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 768, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 768 (delta 1), reused 1 (delta 1), pack-reused 765 (from 1)
Receiving objects: 100% (768/768), 813.61 KiB | 10.71 MiB/s, done.
Resolving deltas: 100% (503/503), done.


In [5]:
%cd /content/ML-Tech

/content/ML-Tech


In [6]:
!git pull origin main

From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Already up to date.


In [9]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.1 MB/s eta 0:00:00


In [10]:
import faiss
index = faiss.read_index("data/processed/passport_index.faiss")

In [11]:
import json

with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

In [12]:
import numpy as np
import pandas as pd
import time

In [13]:
from sentence_transformers import SentenceTransformer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Chunks:", len(chunks))

In [ ]:
embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

In [ ]:
index = faiss.read_index(
    "data/processed/government_index.faiss"
)

print("FAISS vectors:", index.ntotal)
print("Chunks:", len(chunks))

assert index.ntotal == len(chunks)

In [ ]:
def retrieve(question, k=3):
    query_embedding = embedding_model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "document": chunks[idx]["document"],
            "service":chunks[idx].get("service",""),
            "section": chunks[idx]["section"],
            "language":chunks[idx].get("language",""),
            "text": chunks[idx]["text"],
            "url": chunks[idx]["url"]
        })

    return results

In [ ]:
results = retrieve(
    "How much does a 10-year passport cost?",
    k=3
)

for r in results:
    print(
        r["score"],
        "|",
        r["service"],
        "|",
        r["section"]
    )

In [ ]:
test_questions = [
    {
        "question": "How much does a 10-year passport cost?",
        "expected_service": "Biometric Passport",
        "expected_section": "Fees"
    },

    {
        "question": "What should I do if I lose my passport?",
        "expected_service": "Lost or Stolen Passport",
        "expected_section": "Lost Passport"
    },

    {
        "question": "ما هي المستندات المطلوبة لتجديد رخصة سوق منتهية الصلاحية؟",
        "expected_service": "تجديد رخصة سوق منتهية الصلاحة",
        "expected_section": "المستندات المطلوبة"
    }
]

In [ ]:
metrics = evaluate_retrieval(test_questions, k=3)

print("Top-1 Accuracy:", metrics["top1_accuracy"])
print("Recall@3:", metrics["recall@3"])
print("MRR:", metrics["mrr"])